In [1]:
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch, json, re
from datetime import datetime

print("🔄 Loading Qwen...")
model_name = "Qwen/Qwen2-1.5B-Instruct"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.float16,
    device_map="auto"
)
print(f"✅ Qwen loaded on {next(model.parameters()).device}")

# Load databases
with open("data/knowledge_base.json") as f:
    knowledge_base = json.load(f)
with open("data/employees.json") as f:
    employees = json.load(f)

🔄 Loading Qwen...


`torch_dtype` is deprecated! Use `dtype` instead!


✅ Qwen loaded on cuda:0


In [2]:
def call_qwen(user_prompt, system_prompt, max_new_tokens=400):
    """Universal function to call Qwen with any prompt"""
    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt}
    ]
    text = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    inputs = tokenizer([text], return_tensors="pt").to(model.device)
    
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=True,
            temperature=0.2,  # Low = more focused/consistent
            pad_token_id=tokenizer.eos_token_id
        )
    
    generated = outputs[0][inputs['input_ids'].shape[1]:]
    return tokenizer.decode(generated, skip_special_tokens=True).strip()

In [3]:
def agent_analyze_ticket(user_description):
    """
    AGENT 1: Ticket Analyzer
    Takes raw user input → produces structured ticket
    """
    print("🤖 Agent 1 (Analyzer) is working...")
    
    system = """You are an IT Ticket Analyzer. Your job is to analyze IT problems described by business users and create structured tickets.

You MUST respond ONLY with a JSON object — no extra text, no explanation.

The JSON must have exactly these fields:
{
  "title": "short title of the problem (max 10 words)",
  "description": "clear technical description of the issue",
  "category": "bug OR query OR development",
  "priority": "low OR medium OR high OR critical",
  "affected_system": "which system/module is affected",
  "keywords": ["keyword1", "keyword2", "keyword3"],
  "estimated_complexity": "simple OR moderate OR complex"
}

Category rules:
- bug: something is broken, not working, error, crash, failure
- query: asking for data, reports, information, how-to questions  
- development: new feature request, enhancement, integration, automation"""

    prompt = f"""Analyze this IT problem reported by a business user and create a structured ticket:

USER PROBLEM: "{user_description}"

Respond ONLY with the JSON object."""

    response = call_qwen(prompt, system, max_new_tokens=350)
    
    # Extract JSON from response
    try:
        # Try to find JSON in the response
        json_match = re.search(r'\{.*\}', response, re.DOTALL)
        if json_match:
            ticket_data = json.loads(json_match.group())
        else:
            ticket_data = json.loads(response)
            
        # Add metadata
        ticket_data["id"] = f"TKT-{datetime.now().strftime('%Y%m%d%H%M%S')}"
        ticket_data["created_at"] = datetime.now().isoformat()
        ticket_data["status"] = "analyzing"
        ticket_data["original_description"] = user_description
        
        print(f"   ✅ Ticket created: [{ticket_data['id']}] {ticket_data['title']}")
        return ticket_data
        
    except json.JSONDecodeError:
        # Fallback if JSON parsing fails
        print("   ⚠️  Parsing JSON failed, using fallback")
        return {
            "id": f"TKT-{datetime.now().strftime('%Y%m%d%H%M%S')}",
            "title": user_description[:50],
            "description": user_description,
            "category": "bug",
            "priority": "medium",
            "affected_system": "Unknown",
            "keywords": user_description.lower().split()[:5],
            "estimated_complexity": "moderate",
            "created_at": datetime.now().isoformat(),
            "status": "analyzing",
            "original_description": user_description
        }

# TEST IT
test_ticket = agent_analyze_ticket("I cannot log into the system since this morning. It keeps saying invalid password but I haven't changed anything.")
print("\n📋 Generated Ticket:")
print(json.dumps(test_ticket, indent=2))

🤖 Agent 1 (Analyzer) is working...
   ✅ Ticket created: [TKT-20260510082011] Login Issue

📋 Generated Ticket:
{
  "title": "Login Issue",
  "description": "User unable to login due to invalid password after system update.",
  "category": "development",
  "priority": "high",
  "affected_system": "System",
  "keywords": [
    "invalid password",
    "login issue"
  ],
  "estimated_complexity": "complex",
  "id": "TKT-20260510082011",
  "created_at": "2026-05-10T08:20:11.361609",
  "status": "analyzing",
  "original_description": "I cannot log into the system since this morning. It keeps saying invalid password but I haven't changed anything."
}


In [4]:
def agent_search_knowledge_base(ticket):
    """
    AGENT 2: Knowledge Base Searcher
    Searches existing solutions for the ticket
    Returns: solution if found, None if not found
    """
    print("\n🤖 Agent 2 (Knowledge Searcher) is working...")
    
    # Step 1: Simple keyword matching score
    ticket_keywords = [kw.lower() for kw in ticket.get("keywords", [])]
    ticket_text = (ticket["title"] + " " + ticket["description"]).lower()
    
    scored_solutions = []
    for kb_item in knowledge_base:
        score = 0
        
        # Category match (strong signal)
        if kb_item["category"] == ticket.get("category"):
            score += 3
        
        # Keyword overlap
        for kw in kb_item["keywords"]:
            if kw.lower() in ticket_text:
                score += 2
            if kw.lower() in ticket_keywords:
                score += 1
        
        if score > 0:
            scored_solutions.append({**kb_item, "match_score": score})
    
    # Sort by score
    scored_solutions.sort(key=lambda x: x["match_score"], reverse=True)
    
    if not scored_solutions or scored_solutions[0]["match_score"] < 3:
        print("   ℹ️  No strong match found in knowledge base")
        return None
    
    best_match = scored_solutions[0]
    print(f"   ✅ Found potential solution: {best_match['id']} (score: {best_match['match_score']})")
    
    # Step 2: Use AI to verify if solution truly applies
    system = """You are an IT Support Expert. You must decide if a known solution matches a new problem.
Respond ONLY with JSON: {"matches": true or false, "confidence": 0-100, "reason": "brief explanation"}"""
    
    prompt = f"""Does this existing solution solve the new problem?

NEW TICKET:
Title: {ticket['title']}
Description: {ticket['description']}
Category: {ticket['category']}

EXISTING SOLUTION:
Problem it solves: {best_match['problem']}
Solution: {best_match['solution']}

Does this solution apply? Respond ONLY with JSON."""

    ai_response = call_qwen(prompt, system, max_new_tokens=150)
    
    try:
        json_match = re.search(r'\{.*\}', ai_response, re.DOTALL)
        verdict = json.loads(json_match.group() if json_match else ai_response)
        
        if verdict.get("matches") and verdict.get("confidence", 0) > 65:
            print(f"   ✅ AI confirmed match! Confidence: {verdict['confidence']}%")
            return {
                "found": True,
                "solution_id": best_match["id"],
                "solution": best_match["solution"],
                "confidence": verdict["confidence"],
                "reason": verdict.get("reason", ""),
                "success_rate": best_match["success_rate"]
            }
        else:
            print(f"   ℹ️  AI rejected match. Confidence too low: {verdict.get('confidence', 0)}%")
            return None
            
    except:
        # If AI parsing fails, trust the keyword score
        if best_match["match_score"] >= 5:
            return {
                "found": True,
                "solution_id": best_match["id"],
                "solution": best_match["solution"],
                "confidence": 70,
                "reason": "Keyword match",
                "success_rate": best_match["success_rate"]
            }
        return None

# TEST IT
kb_result = agent_search_knowledge_base(test_ticket)
print("\n🔍 Knowledge Base Result:")
print(json.dumps(kb_result, indent=2))


🤖 Agent 2 (Knowledge Searcher) is working...
   ✅ Found potential solution: KB001 (score: 4)
   ✅ AI confirmed match! Confidence: 95%

🔍 Knowledge Base Result:
{
  "found": true,
  "solution_id": "KB001",
  "solution": "1. Clear browser cache and cookies\n2. Reset password via /forgot-password\n3. Check if account is locked in Active Directory\n4. If still failing, check VPN connection and retry",
  "confidence": 95,
  "reason": "The solution provided addresses the issue of user inability to login due to invalid password after a system update, which is the same as the new problem described.",
  "success_rate": 0.92
}


In [5]:
def agent_find_best_employee(ticket):
    """
    AGENT 3: Smart Employee Assigner
    Finds the best available employee for this ticket
    """
    print("\n🤖 Agent 3 (Employee Assigner) is working...")
    
    ticket_category = ticket.get("category", "bug")
    ticket_keywords = [kw.lower() for kw in ticket.get("keywords", [])]
    ticket_complexity = ticket.get("estimated_complexity", "moderate")
    
    # Score each employee
    scored_employees = []
    
    for emp in employees:
        # Skip fully loaded employees
        if emp["current_tickets"] >= emp["max_capacity"]:
            print(f"   ⏭️  Skipping {emp['name']} — fully loaded")
            continue
        
        score = 0
        
        # 1. Availability (most important)
        score += emp["availability_score"] * 40  # Max 40 points
        
        # 2. Category specialization
        if ticket_category in emp["specialization"]:
            score += 25
        
        # 3. Skill keyword match
        emp_skills_text = " ".join(emp["skills"]).lower()
        for kw in ticket_keywords:
            if kw in emp_skills_text:
                score += 5  # 5 points per matching keyword
        
        # 4. Experience bonus for complex tickets
        if ticket_complexity == "complex" and emp["experience_years"] > 6:
            score += 10
        
        scored_employees.append({
            "employee": emp,
            "score": round(score, 2)
        })
    
    if not scored_employees:
        print("   ❌ No available employees found!")
        return None
    
    # Sort by score
    scored_employees.sort(key=lambda x: x["score"], reverse=True)
    
    # Show top candidates
    print("   📊 Top candidates:")
    for i, candidate in enumerate(scored_employees[:3]):
        emp = candidate["employee"]
        print(f"      {i+1}. {emp['name']} — Score: {candidate['score']} | Free slots: {emp['max_capacity'] - emp['current_tickets']}")
    
    best = scored_employees[0]["employee"]
    
    # Use AI to write a professional assignment message
    system = "You are an IT Manager. Write a brief, professional ticket assignment message. Be concise (3-4 sentences max)."
    
    prompt = f"""Write an assignment notification for this IT ticket:

Ticket: {ticket['title']}
Category: {ticket['category']}  
Priority: {ticket['priority']}
Assigned to: {best['name']} ({best['role']})
Their relevant skills: {', '.join(best['skills'][:4])}

Write a brief professional message telling them they've been assigned this ticket."""

    assignment_message = call_qwen(prompt, system, max_new_tokens=150)
    
    print(f"\n   ✅ Best match: {best['name']} (Score: {scored_employees[0]['score']})")
    
    return {
        "assigned_to": best["name"],
        "employee_id": best["id"],
        "email": best["email"],
        "role": best["role"],
        "assignment_score": scored_employees[0]["score"],
        "current_load": f"{best['current_tickets']}/{best['max_capacity']} tickets",
        "assignment_message": assignment_message,
        "top_3_candidates": [
            {
                "name": c["employee"]["name"],
                "role": c["employee"]["role"],
                "score": c["score"]
            } for c in scored_employees[:3]
        ]
    }

# TEST IT
assignment = agent_find_best_employee(test_ticket)
print("\n👤 Assignment Result:")
print(json.dumps(assignment, indent=2))


🤖 Agent 3 (Employee Assigner) is working...
   📊 Top candidates:
      1. Vikram Singh — Score: 65.0 | Free slots: 4
      2. Arjun Sharma — Score: 59.0 | Free slots: 3
      3. Sneha Nair — Score: 57.0 | Free slots: 4

   ✅ Best match: Vikram Singh (Score: 65.0)

👤 Assignment Result:
{
  "assigned_to": "Vikram Singh",
  "employee_id": "EMP005",
  "email": "vikram.singh@company.com",
  "role": "DevOps Engineer",
  "assignment_score": 65.0,
  "current_load": "0/4 tickets",
  "assignment_message": "Dear Vikram Singh,\n\nI am writing to inform you that you have been assigned the login issue ticket from our system. This is a high priority issue and your expertise in Docker, Kubernetes, CI/CD, and monitoring will be invaluable in resolving it.\n\nPlease let me know if there's anything else I can do to assist you with this task. Thank you for your attention to this matter.\n\nBest regards,\n[Your Name]",
  "top_3_candidates": [
    {
      "name": "Vikram Singh",
      "role": "DevOps Engin

In [6]:
def agent_present_solution(ticket, kb_result):
    """
    AGENT 4: Solution Presenter
    Takes a KB solution and makes it user-friendly
    """
    print("\n🤖 Agent 4 (Solution Presenter) is working...")
    
    system = """You are a friendly IT Support assistant. 
Present the solution clearly and professionally.
Format it nicely with numbered steps.
End with asking if this solved their problem."""

    prompt = f"""A business user has this IT problem:
"{ticket['original_description']}"

Here is the solution from our knowledge base:
{kb_result['solution']}

Write a friendly, clear response that:
1. Acknowledges their problem briefly  
2. Presents the solution with clear numbered steps
3. Mentions the success rate is {int(kb_result['success_rate']*100)}% for similar issues
4. Asks them to confirm if this resolved their issue"""

    response = call_qwen(prompt, system, max_new_tokens=400)
    return response

# TEST - present a solution
if kb_result:
    solution_message = agent_present_solution(test_ticket, kb_result)
    print("\n💬 Solution Message for User:")
    print("-" * 50)
    print(solution_message)


🤖 Agent 4 (Solution Presenter) is working...

💬 Solution Message for User:
--------------------------------------------------
I'm sorry to hear you're having trouble logging into your system. Here's how we can help:

1. **Clear Browser Cache and Cookies**: Try clearing your browser cache and cookies. This might resolve any temporary issues or security settings that could be preventing access.

2. **Reset Password via /forgot-password**: If the issue persists after clearing your cache, try resetting your password by clicking on the "Forgot Password" link at the bottom of the login page. This will send you an email with instructions to reset your password.

3. **Check if Account is Locked in Active Directory**: If you've tried both methods above and are still unable to log in, there may be an issue with your account being locked out due to incorrect passwords or other reasons. You'll need to contact IT support for assistance with unlocking your account.

4. **If Still Failing, Check VPN